In [1]:
from argparse import Namespace
import yaml

with open('Pretrained MoLFormer/hparams.yaml', 'r') as f:
    config = Namespace(**yaml.safe_load(f))
config


Namespace(accelerator='ddp', batch_size=64, beam_size=0, checkpoint_every=5000, clip_grad=50, config_load=None, config_save=None, d_dropout=0.2, data_path='', data_root='/dccstor/medscan7/smallmolecule/runs/ba-predictor/small-data/affinity', dataset_length=None, dataset_name='sol', debug=True, device='cuda', dropout=0.1, eval_every=1000, fast_dev_run=False, fc_h=512, finetune_path='', freeze_model=False, from_scratch=False, gen_save=None, gpus=8, grad_acc=1, log_file=None, lr=0.001, lr_end=0.00030000000000000003, lr_multiplier=8, lr_start=3e-05, max_epochs=4, max_len=202, measure_name='measure', min_len=1, mode='cls', model_arch='BERT_16GPU_Both_10percent_rotate_no_masking', model_load=None, model_save='model.pt', model_save_dir='./models_dump/', n_batch=1800, n_embd=768, n_head=12, n_jobs=1, n_last=1000, n_layer=12, n_samples=None, n_workers=8, nucleus_thresh=0.9, num_epoch=1, num_feats=32, num_nodes=1, num_seq_returned=0, num_workers=0, pretext_size=0, q_dropout=0.5, restart_path='',

In [2]:
from tokenizer.tokenizer import MolTranBertTokenizer

tokenizer = MolTranBertTokenizer('bert_vocab.txt')
tokenizer.vocab

OrderedDict([('<bos>', 0),
             ('<eos>', 1),
             ('<pad>', 2),
             ('<mask>', 3),
             ('C', 4),
             ('c', 5),
             ('(', 6),
             (')', 7),
             ('1', 8),
             ('O', 9),
             ('N', 10),
             ('2', 11),
             ('=', 12),
             ('n', 13),
             ('3', 14),
             ('[C@H]', 15),
             ('[C@@H]', 16),
             ('F', 17),
             ('S', 18),
             ('4', 19),
             ('Cl', 20),
             ('-', 21),
             ('o', 22),
             ('s', 23),
             ('[nH]', 24),
             ('#', 25),
             ('/', 26),
             ('Br', 27),
             ('[C@]', 28),
             ('[C@@]', 29),
             ('[N+]', 30),
             ('[O-]', 31),
             ('5', 32),
             ('\\', 33),
             ('.', 34),
             ('I', 35),
             ('6', 36),
             ('[S@]', 37),
             ('[S@@]', 38),
             ('P', 39)

In [3]:
from train_pubchem_light import LightningModule

ckpt = 'Pretrained MoLFormer/checkpoints/N-Step-Checkpoint_3_30000.ckpt'
lm = LightningModule(config, tokenizer.vocab).load_from_checkpoint(ckpt, config=config, vocab=tokenizer.vocab)
lm



`fused_weight_gradient_mlp_cuda` module not found. gradient accumulation fusion with weight gradient computation disabled.
Global seed set to 12345


Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


Global seed set to 12345


Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


LightningModule(
  (tok_emb): Embedding(2362, 768)
  (drop): Dropout(p=0.2, inplace=False)
  (blocks): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (attention): RotateAttentionLayer(
          (inner_attention): LinearAttention(
            (feature_map): GeneralizedRandomFeatures()
          )
          (query_projection): Linear(in_features=768, out_features=768, bias=True)
          (key_projection): Linear(in_features=768, out_features=768, bias=True)
          (value_projection): Linear(in_features=768, out_features=768, bias=True)
          (out_projection): Linear(in_features=768, out_features=768, bias=True)
          (rotaryemb): RotaryEmbedding()
        )
        (linear1): Linear(in_features=768, out_features=768, bias=True)
        (linear2): Linear(in_features=768, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=

In [4]:
import torch
from fast_transformers.masking import LengthMask as LM

def batch_split(data, batch_size=64):
    i = 0
    while i < len(data):
        yield data[i:min(i+batch_size, len(data))]
        i += batch_size

def embed(model, smiles, tokenizer, batch_size=64):
    model.eval()
    embeddings = []
    for batch in batch_split(smiles, batch_size=batch_size):
        batch_enc = tokenizer.batch_encode_plus(batch, padding=True, add_special_tokens=True)
        idx, mask = torch.tensor(batch_enc['input_ids']), torch.tensor(batch_enc['attention_mask'])
        with torch.no_grad():
            token_embeddings = model.blocks(model.tok_emb(idx), length_mask=LM(mask.sum(-1)))
        # average pooling over tokens
        input_mask_expanded = mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embedding = sum_embeddings / sum_mask
        embeddings.append(embedding.detach().cpu())
    return torch.cat(embeddings)



In [6]:
import pandas as pd

df = pd.read_csv('../../../E Choli/data/clean_data.csv', index_col=0)
# df.dropna(subset=['curated_smiles'], inplace=True)
df


,SMILES,Qualifier,EC50
PUBCHEM_RESULT_TAG,,,
3,C1=CC=C(C=C1)C2=NC3=CC=CC=C3C(=N2)SCC(=O)NC4=N...,=,45.000
7,C1=CC=C2C(=C1)C=CC(=N2)C(=O)C3=NN[C@H]4[C@@H]3...,=,0.271
16,CC1=CC=C(C=C1)C(=O)OC2=NN(C(=C2)N)S(=O)(=O)C3=...,=,4.100
20,CC1(C2=C(C=CC(=C2)S(=O)(=O)NCC3=CC=C(C=C3)C(=O...,=,29.200
24,COC(=O)C1=NN2C(=CC(=NC2=C1Br)C3=CC=C(O3)Br)C(F...,=,30.500
...,...,...,...
455,CC1=C(C=C(C=C1)S(=O)(=O)C2=CC3=C(N=C4C=CC=CN4C...,=,52.200
459,C1=CC=C2C(=C1)NC(=N2)/C(=C(/C3=CC=C(O3)[N+](=O...,=,5.530
460,C1=CC=C(C=C1)CC2=C[N+](=CC=C2)C\3=C(C4=CC=CC=C...,=,36.500


In [10]:
from rdkit import Chem

def canonicalize(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True, isomericSmiles=False)

smiles = df.SMILES.apply(canonicalize)
X = embed(lm, smiles, tokenizer).numpy()
y = df.EC50

X



array([[ 0.9863817 ,  0.01068352,  1.1602038 , ...,  0.29371312,
        -0.2003039 , -0.32942042],
       [-0.1775578 ,  0.06196711,  0.6559694 , ...,  0.00591139,
         0.31482652, -0.5996463 ],
       [ 0.1390997 ,  0.22166221,  0.34596696, ..., -0.31077513,
        -0.27373785, -0.02017703],
       ...,
       [ 0.7934927 ,  0.05513205,  0.8375609 , ...,  0.61616117,
        -0.7399386 , -0.47408643],
       [-0.10710429,  0.15216762,  1.1010753 , ...,  0.37111032,
         0.06500692,  0.23024386],
       [-0.00197919,  0.31063917,  0.52428985, ...,  0.5572159 ,
         0.18280733, -0.08628924]], dtype=float32)

In [11]:
embeddings_df = pd.DataFrame(X, index=df.index)
embeddings_df['EC50'] = y
embeddings_df.to_csv('../../../E Choli/data/MolFormer_embeddings.csv')
# final_df = pd.concat([df[['target_organism', 'train', 'curated_smiles', 'pIC50']], embeddings_df], axis=1)
# final_df.to_csv('../../../data/MolFormer_mean.csv')